In [ ]:
%%capture
import torch
major_version, minor_version = torch.cuda.get_device_capability()
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
if major_version >= 8:
    !pip install --no-deps packaging ninja einops "flash-attn>=2.6.3" xformers trl peft accelerate bitsandbytes
else:
    !pip install --no-deps xformers trl peft accelerate bitsandbytes

In [ ]:
!nvidia-smi

Sat Aug 15 10:17:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from unsloth import FastLanguageModel
from transformers import TextStreamer
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
import torch

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
# Load the base model with 4-bit quantization
max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
)

==((====))==  Unsloth 2026.8.18: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


In [ ]:
# Test the base model BEFORE fine-tuning
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": "You are a helpful customer support assistant."},
    {"role": "user", "content": "I need help canceling my order"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

text_streamer = TextStreamer(tokenizer, skip_prompt=True)
print("=== BASE MODEL RESPONSE (before fine-tuning) ===")
_ = model.generate(
    input_ids=inputs,
    streamer=text_streamer,
    max_new_tokens=256,
    temperature=0.7,
    do_sample=True,
)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


=== BASE MODEL RESPONSE (before fine-tuning) ===


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


I'd be happy to assist you with canceling your order. Can you please provide me with some details?

* Your order number (if you have it)
* The date you placed the order
* A brief explanation of why you'd like to cancel the order

This will help me look into your order and assist you with the cancellation process.<|eot_id|>


In [ ]:
# Load the Bitext customer support dataset
dataset = load_dataset(
    "bitext/Bitext-customer-support-llm-chatbot-training-dataset",
    split="train",
)
print(f"Dataset size: {len(dataset)} examples")
print(f"Columns: {dataset.column_names}")
print(f"\nExample:")
print(f"Category: {dataset[0]['category']}")
print(f"Intent: {dataset[0]['intent']}")
print(f"Instruction: {dataset[0]['instruction']}")
print(f"Response: {dataset[0]['response'][:200]}...")

README.md:   0%|          | 0.00/11.9k [00:00<?, ?B/s]

Bitext_Sample_Customer_Support_Training_(…): reconstructing file:   0%|          |  0.00B / 19.2MB            

Bitext_Sample_Customer_Support_Training_(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/26872 [00:00<?, ? examples/s]

Dataset size: 26872 examples
Columns: ['flags', 'instruction', 'category', 'intent', 'response']

Example:
Category: ORDER
Intent: cancel_order
Instruction: question about cancelling order {{Order Number}}
Response: I've understood you have a question regarding canceling order {{Order Number}}, and I'm here to provide you with the information you need. Please go ahead and ask your question, and I'll do my best to...


In [ ]:
# Format dataset for chat-style fine-tuning
def format_chat(example):
    messages = [
        {"role": "system", "content": "You are a helpful customer support assistant."},
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": example["response"]},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

dataset = dataset.map(format_chat)
print("Formatted example:")
print(dataset[0]["text"][:500])

Map:   0%|          | 0/26872 [00:00<?, ? examples/s]

Formatted example:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 15 Aug 2026

You are a helpful customer support assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>

question about cancelling order {{Order Number}}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

I've understood you have a question regarding canceling order {{Order Number}}, and I'm here to provide you with the information you need. Please go ahead and ask your questi


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    max_seq_length=max_seq_length,
)

Unsloth 2026.8.18 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [ ]:
model.print_trainable_parameters()

trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511


In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    tokenizer=tokenizer,
    args=SFTConfig(
        max_seq_length=max_seq_length,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=60,
        learning_rate=2e-4,
        logging_steps=1,
        output_dir="outputs",
        optim="adamw_8bit",
        seed=3407,
    ),
)

trainer.train()

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/26872 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 26,872 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,2.548348
2,2.297602
3,2.410326
4,2.462442
5,2.391105
6,2.358824
7,2.155732
8,2.281674
9,1.935930
10,1.630923


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.


TrainOutput(global_step=60, training_loss=1.2650546530882518, metrics={'train_runtime': 176.9894, 'train_samples_per_second': 2.712, 'train_steps_per_second': 0.339, 'total_flos': 1474184118312960.0, 'train_loss': 1.2650546530882518, 'epoch': 0.017862459065197976})

In [16]:
# Switch the model back to fast inference mode
FastLanguageModel.for_inference(model)

# Four questions covering different customer support intents
test_questions = [
    "I need help canceling my order",
    "What is your refund policy?",
    "How do I change my shipping address?",
    "I want to delete my account",
]

# Test each question and stream the model's response
for question in test_questions:
    messages = [
        {"role": "system", "content": "You are a helpful customer support assistant."},
        {"role": "user", "content": question},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")

    text_streamer = TextStreamer(tokenizer, skip_prompt=True)
    print(f"\n{'='*50}")
    print(f"Question: {question}")
    print(f"{'='*50}")
    _ = model.generate(
        input_ids=inputs,
        streamer=text_streamer,
        max_new_tokens=256,
        temperature=0.7,
        do_sample=True,
    )

Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Question: I need help canceling my order
I'm sorry to hear that you need assistance with canceling your order. I'm here to help you through the process. To cancel your order, please provide me with the following information: 
1. Your order number
2. The reason for cancellation

Once I have this information, I'll be able to guide you through the steps to complete the cancellation. Your satisfaction is our top priority, and I'm committed to ensuring that you have a smooth and hassle-free experience. Thank you for your patience and cooperation!<|eot_id|>

Question: What is your refund policy?


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


I'm glad you're interested in learning about our refund policy! At {{Company Name}}, we strive to provide you with a seamless and hassle-free experience. Our refund policy is designed to ensure that you are satisfied with your purchase and can easily return or exchange any item that does not meet your expectations.

Here are the general steps and guidelines for our refund policy:

1. {{Step 1 Description}}
2. {{Step 2 Description}}
3. {{Step 3 Description}}

If you have any further questions or need additional assistance, please don't hesitate to let me know. I'm here to help you navigate our refund policy and provide any additional information you may need.<|eot_id|>


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Question: How do I change my shipping address?
I appreciate your interest in updating your shipping address. To ensure a seamless and efficient process, I recommend that you log in to your account on our website and navigate to the "Account Settings" or "Order History" section. From there, you should be able to locate the option to edit your shipping address. If you encounter any difficulties or have further questions, please don't hesitate to let me know. Your satisfaction is our top priority, and I'm here to assist you every step of the way.<|eot_id|>


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Question: I want to delete my account
I'm truly sorry to hear that you're considering deleting your account. I understand how important it is for you to have control over your personal information and online presence. Rest assured that I'm here to assist you in any way I can. To proceed with deleting your account, could you please provide me with some more details? This will help me ensure that we follow the correct procedures and take any necessary steps to ensure your account is deleted safely and efficiently. Your feedback is invaluable, and I appreciate your trust in our platform.<|eot_id|>


In [ ]:
# Save the LoRA adapter and tokenizer to a local directory
model.save_pretrained("customer_support_lora")
tokenizer.save_pretrained("customer_support_lora")
print("Model saved! LoRA adapter is ~100MB compared to the full 6GB+ model.")

Unsloth: Restored added_tokens_decoder metadata in customer_support_lora/tokenizer_config.json.


Model saved! LoRA adapter is ~100MB compared to the full 6GB+ model.


In [ ]:
from huggingface_hub import login

# Paste your Hugging Face write access token here
login(token="your-token-here")

hub_model_id = "Jack217/customer-support-llama-3.2-3b-lora"

model.push_to_hub(hub_model_id)
tokenizer.push_to_hub(hub_model_id)

print(f"Model pushed to https://huggingface.co/{hub_model_id}")

README.md:   0%|          | 0.00/581 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 45.8kB / 97.3MB            

Saved model to https://huggingface.co/Jack217/customer-support-llama-3.2-3b-lora


Unsloth: Restored added_tokens_decoder metadata in /tmp/tmpeq1007s4/tokenizer_config.json.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpeq1007s4/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

Model pushed to https://huggingface.co/Jack217/customer-support-llama-3.2-3b-lora
